In [1]:
import mediapipe as mp
import cv2
import numpy as np
from PIL import Image

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    running_mode=VisionRunningMode.IMAGE
)

def get_blendshapes(image_path):
    img = Image.open(image_path).convert('RGB')
    img_rgb = np.array(img)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    
    with FaceLandmarker.create_from_options(options) as landmarker:
        result = landmarker.detect(mp_image)
    
    if not result.face_blendshapes:
        return None
    
    return {b.category_name: b.score for b in result.face_blendshapes[0]}

def classify_emotion(bs):
    smile       = (bs['mouthSmileLeft']  + bs['mouthSmileRight'])  / 2
    eye_squint  = (bs['eyeSquintLeft']   + bs['eyeSquintRight'])   / 2
    brow_down   = (bs['browDownLeft']    + bs['browDownRight'])    / 2
    mouth_frown = (bs['mouthFrownLeft']  + bs['mouthFrownRight'])  / 2
    eye_wide    = (bs['eyeWideLeft']     + bs['eyeWideRight'])     / 2

    if smile > 0.25 and eye_squint > 0.3:
        return 'happy'
    
    elif mouth_frown > 0.3 or (brow_down > 0.4 and eye_wide < 0.1):
        return 'sad'
    
    else:
        return 'neutral'

def analyze_image(image_path):
    bs = get_blendshapes(image_path)
    if bs is None:
        print("얼굴 감지 안됨")
        return
    
    emotion = classify_emotion(bs)
    
    print(f"감정: {emotion}")
    print(f"--- 주요 수치 ---")
    print(f"mouthSmile: {(bs['mouthSmileLeft']+bs['mouthSmileRight'])/2:.4f}")
    print(f"eyeSquint: {(bs['eyeSquintLeft']+bs['eyeSquintRight'])/2:.4f}")
    print(f"browDown: {(bs['browDownLeft']+bs['browDownRight'])/2:.4f}")
    print(f"mouthFrown: {(bs['mouthFrownLeft']+bs['mouthFrownRight'])/2:.4f}")
    print(f"cheekSquint: {(bs.get('cheekSquintLeft',0)+bs.get('cheekSquintRight',0))/2:.4f}")
    
    return emotion

# 테스트
analyze_image('/workspace/frames/frame_00016.jpg')

2026-04-14 11:04:19.495976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/frames/frame_00016.jpg'